# Train CLIP Model

Before training the CLIP model, we need to prepare the configuration file. This includes settings for data augmentation, spectrum and peptide tokenizers, model architecture sizes, optimizers, learning rate schedulers, trainer setups, and data paths.

Below is a detailed introduction to the meaning and usage of the main modules and parameters in the configuration file:

## aug (Data Augmentation)

Controls dynamic augmentation operations on mass spectrometry data during training to improve the model's generalization ability.

-   `enabled`: Set to true to enable data augmentation.
-   `return_dummy_tensor`: Whether to return dummy data, which can partially improve performance when dealing with small batch sizes.
-   `prob`: The overall probability of triggering data augmentation (e.g., 0.5 means 50% of the data will be augmented).
-   `removal_rate` / `perturbation_rate`: Controls the random removal ratio and the intensity perturbation ratio of the peaks, respectively.
-   `removal_intensity_threshold`: The intensity threshold for the removal operation, ensuring that high-intensity critical peaks are not accidentally removed.

## tokenizer (Data Digitization)

Responsible for converting raw mass spectra and peptide strings into tensors that the model can process.

### Spectrum

We recommend the following two classic configurations:

Configuration A: 
-   `n_top_peaks`: 150
-   `min_mz`: 50.0
-   `max_mz`: 2500.0

Configuration B (Wide Range, as shown in the example):
-   `n_top_peaks`: 300
-   `min_mz`: 50.0
-   `max_mz`: 4500.0

The configuration B can get better performance.

-   `min_intensity` / `remove_precursor_tol`: Used to filter out low-intensity noise peaks and remove precursor ion residual windows.

### Peptide

-   `reverse`: Whether to reverse the peptide sequence during tokenization. 
-   `residues`: Specifies the amino acid residue vocabulary and Post-Translational Modifications (PTMs). You have three ways to configure this:
    -   Custom Dictionary: You can pass a complete dictionary mapping residue strings to their exact mass values.
    -   "canonical": Uses the standard 20 amino acids. Note that Cysteine (C) still carries a fixed carbamidomethylation modification by default (C+57.021).
    -   "massivekb": Includes the canonical amino acids plus common PTMs found in the MassIVE-KB dataset. This adds N-terminal modifications (Acetylation +42.011, Carbamylation +43.006, NH3 loss -17.027) and specific amino acid modifications (Met Oxidation M+15.995, Asn/Gln Deamidation N+0.984, Q+0.984).

## model (Model Architecture)
Defines the network scale for the spectrum encoder and peptide encoder in the dual-tower Contrastive Language-Image Pre-training (CLIP) model.

-   `hidden_size` / `n_head` / `n_layers` / `dim_feedforward`: Core hyperparameters for the Transformer encoders and decoders.
-   `dropout`: The dropout rate to prevent overfitting.
-   `logits_scale` / `learnable_logits_scale`:  Temperature coefficient scaling configuration for the contrastive learning loss function.

## optimizer

-   `lr`: The base learning rate (e.g., 1.0e-4).
-   `weight_decay`: The L2 regularization/weight decay coefficient used to prevent overfitting.

## scheduler (Learning Rate Scheduler)

Used to dynamically adjust the learning rate during the training process.
-   `enabled`: Whether to enable the scheduler.
-   `warmup_steps`: The number of warmup steps. This parameter is highly flexible:
    You can provide a float less than 1.0 (e.g., 0.1), and the system will automatically multiply it by the total training steps to calculate the dynamic warmup step percentage.

    You can also provide an integer greater than 1 (e.g., 9500 as in the example) to force a specific, absolute number of warmup steps.

-   `n_cycles`: The number of cosine cycles (restarts) over the entire training process. If set to 1 (Standard), the learning rate follows a single, smooth cosine decay curve down to zero after the warmup. If set to > 1 (e.g., 3), the learning rate will "restart" to a peak value periodically, which can help the model escape local minima.
-   `lr_decay_factor`: The decay multiplier applied to the peak learning rate at each restart. For example, if this is 5.0 and n_cycles is 3, the peak learning rate of the 2nd cycle will be $1/5$ (20%) of the base learning rate, and the 3rd cycle will be $1/25$ (4%).
    -   ⚠️ Important Note: This parameter strictly controls the peak decay between cycles. Therefore, if n_cycles is set to 1, lr_decay_factor becomes an inactive dummy variable, and its value (whether 1.0 or 10.0) will have no effect on the final learning rate curve.

## trainer

Core training control parameters based on PyTorch Lightning.

-   `devices`: Specifies which GPUs to use for training. For example, passing the list [0, 1] enables distributed training on GPU 0 and GPU 1.
-   `task_name` / `evaluate_metric_name` / `is_max`: Sets the current task name (e.g., "clip") and the metric used to evaluate model performance (e.g., "top1_accuracy", where is_max: true indicates that a higher value is better).
-   `save_top_k`: Automatically saves the top K performing checkpoints.
-   `max_epochs` / `validation_steps`: Controls the total number of training epochs and the frequency of validation checks.
-   `summarywriter_folder` / `model_save_folder`:  Sets the paths for TensorBoard logs and checkpoint weight files. (Note: Please be sure to change these to your actual local paths.)

## data

-   `train_path` / `val_path`: Paths to the preprocessed HDF5 datasets.
    you can also set the test_path to `""`.
-   `train_batch_size` / `val_batch_size`: The batch size for each stage (train/val/test).
-   `n_workers`: The number of dataloader worker processes.

The `test_path` and `test_batch_size` can be left unset. We don't use them.

In [5]:
from ruamel.yaml import YAML

# replace the data path with your own path

yaml = YAML()
yaml.preserve_quotes = True
yaml.indent(mapping=2, sequence=4, offset=2)

config = """
aug:
  enabled: true
  prob: 0.5
  removal_rate: 0.2
  removal_intensity_threshold: 0.3
  perturbation_rate: 0.15
  return_dummy_tensor: false

tokenizer:
  spectrum:
    n_top_peaks: 300
    min_mz: 50.0
    max_mz: 4500.0
    min_intensity: 0.01
    remove_precursor_tol: 2.0
  
  peptide:
    reverse: true
    residues: "massivekb"

model:
  spectrum:
    hidden_size: 512
    n_head: 8
    n_layers: 9
    dropout: 0.18
    dim_feedforward: 1024
  
  peptide:
    n_vocab: 29
    hidden_size: 512
    n_head: 8
    n_layers: 9
    dropout: 0.18
    dim_feedforward: 1024
  
  logits_scale: 0.05
  learnable_logits_scale: true

optimizer:
  lr: 1.0e-4
  weight_decay: 1.0e-4

scheduler:
  enabled: true
  n_cycles: 1
  warmup_steps: 9500
  lr_decay_factor: 1.0

trainer:
  is_max: true
  evaluate_metric_name: "top1_accuracy"
  task_name: "clip"
  random_seed: 4545
  save_top_k: 3
  max_epochs: 10
  devices: [0, 1]
  grad_norm_clip: 1.5
  validation_steps: 40000
  show_progress_bar: true
  grad_scaler_enable: true
  distributed: "ddp"
  gradient_accumulation_steps: 1
  summarywriter_folder: "./outputs/tb"
  model_save_folder: "./outputs/checkpoints/clip"

data:
  train_path: ""
  val_path: ""
  test_path: ""
  train_batch_size: 300
  val_batch_size: 300
  test_batch_size: 256
  n_workers: 16
"""

config = yaml.load(config)
yaml.dump(config, open("/data2/xp/RocNovo-Lightning/outputs/clip.yaml", "w"))

In [ ]:
import sys
sys.path.append("..")

from rocnovo.module.clip import train as train_clip

train_clip("/data2/xp/RocNovo-Lightning/outputs/clip.yaml")